In [432]:
import geopandas as gpd
import pandas as pd
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
from kneed import KneeLocator

pd.set_option('display.max_columns', None)

In [433]:
raw_npn = gpd.read_file("../data/raw/climate_matched_phenometrics.geojson")
trees = gpd.read_file("../data/processed/philly_trees.geojson")


In [434]:
individuals = raw_npn.groupby("scientific_name", as_index=False)["Individual_ID"].agg(['nunique', 'count'])
raw_npn

,Site_ID,Latitude,Longitude,Elevation_in_Meters,State,Species_ID,Genus,Species,Common_Name,Kingdom,Individual_ID,Phenophase_ID,Phenophase_Description,First_Yes_Year,First_Yes_Month,First_Yes_Day,First_Yes_DOY,First_Yes_Julian_Date,NumDays_Since_Prior_No,Last_Yes_Year,Last_Yes_Month,Last_Yes_Day,Last_Yes_DOY,Last_Yes_Julian_Date,NumDays_Until_Next_No,index_right,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,scientific_name,geometry
0,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,205,Open flowers (lilac),2010,4,7,97,2455294,1,2010,4,12,102,2455299,1,3192,24,003,01710958,0500000US24003,24003,Anne Arundel,Anne Arundel County,MD,Maryland,06,1074353889,448032843,Syringa chinensis,POINT (-76.55337 38.88813)
1,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,206,Full flowering (lilac),2010,4,13,103,2455300,1,2010,4,19,109,2455306,7,3192,24,003,01710958,0500000US24003,24003,Anne Arundel,Anne Arundel County,MD,Maryland,06,1074353889,448032843,Syringa chinensis,POINT (-76.55337 38.88813)
2,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,207,End of flowering (lilac/honeysuckle),2010,4,26,116,2455313,7,2010,4,28,118,2455315,2,3192,24,003,01710958,0500000US24003,24003,Anne Arundel,Anne Arundel County,MD,Maryland,06,1074353889,448032843,Syringa chinensis,POINT (-76.55337 38.88813)
3,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,373,Breaking leaf buds (lilac/honeysuckle),2010,3,22,81,2455278,7,2010,3,22,81,2455278,4,3192,24,003,01710958,0500000US24003,24003,Anne Arundel,Anne Arundel County,MD,Maryland,06,1074353889,448032843,Syringa chinensis,POINT (-76.55337 38.88813)
4,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,374,All leaf buds broken (lilac/honeysuckle),2010,3,26,85,2455282,4,2010,4,30,120,2455317,-9999,3192,24,003,01710958,0500000US24003,24003,Anne Arundel,Anne Arundel County,MD,Maryland,06,1074353889,448032843,Syringa chinensis,POINT (-76.55337 38.88813)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27158,56856,40.052708,-75.687523,134,PA,1431,Amelanchier,arborea,common serviceberry,Plantae,375036,500,Flowers or flower buds,2026,4,8,98,2461139,-9999,2026,4,14,104,2461145,5,2347,42,029,01209174,0500000US42029,42029,Chester,Chester County,PA,Pennsylvania,06,1943905797,22412288,Amelanchier arborea,POINT (-75.68752 40.05271)
27159,56856,40.052708,-75.687523,134,PA,1431,Amelanchier,arborea,common serviceberry,Plantae,375036,501,Open flowers,2026,4,14,104,2461145,6,2026,4,14,104,2461145,5,2347,42,029,01209174,0500000US42029,42029,Chester,Chester County,PA,Pennsylvania,06,1943905797,22412288,Amelanchier arborea,POINT (-75.68752 40.05271)
27160,59544,38.996899,-77.112534,-9999,MD,195,Asclepias,tuberosa,butterfly milkweed,Plantae,375264,482,Initial growth (forbs),2026,4,12,102,2461143,2,2026,4,14,104,2461145,2,1845,24,031,01712500,0500000US24031,24031,Montgomery,Montgomery County,MD,Maryland,06,1277193339,35686502,Asclepias tuberosa,POINT (-77.11253 38.9969)
27161,59544,38.996899,-77.112534,-9999,MD,195,Asclepias,tuberosa,butterfly milkweed,Plantae,375264,488,Leaves (forbs),2026,4,16,106,2461147,2,2026,4,20,110,2461151,-9999,1845,24,031,01712500,0500000US24031,24031,Montgomery,Montgomery County,MD,Maryland,06,1277193339,35686502,Asclepias tuberosa,POINT (-77.11253 38.9969)


In [435]:
over_25_individuals = individuals.loc[individuals["nunique"] >= 25]
over_25_obs = individuals.loc[individuals["count"] >= 25]


In [436]:
philly_trees_over25_obs = trees.loc[trees["scientific_name"].isin(over_25_obs["scientific_name"])]
philly_trees_over25_obs

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry
5,6,Ilex opaca - american holly,Ilex opaca,american holly,Ilex,opaca,POINT (-75.21047 39.98401)
10,12,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05038 40.0622)
11,13,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05045 40.06207)
12,14,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05082 40.06179)
13,15,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05089 40.06166)
...,...,...,...,...,...,...,...
150025,151717,Gleditsia triacanthos - honeylocust,Gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.0698 40.07192)
150026,151718,Gleditsia triacanthos - honeylocust,Gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.06968 40.0719)
150032,151724,Gleditsia triacanthos - honeylocust,Gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16226 39.99548)
150033,151725,Gleditsia triacanthos - honeylocust,Gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16221 39.9955)


In [437]:
philly_trees_over25_ind = trees.loc[trees["scientific_name"].isin(over_25_individuals["scientific_name"])]
philly_trees_over25_ind

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry
10,12,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05038 40.0622)
11,13,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05045 40.06207)
12,14,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05082 40.06179)
13,15,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.05089 40.06166)
14,16,Liriodendron tulipifera - tulip tree,Liriodendron tulipifera,tulip tree,Liriodendron,tulipifera,POINT (-75.03537 40.04985)
...,...,...,...,...,...,...,...
149986,151678,Cercis canadensis - eastern redbud,Cercis canadensis,eastern redbud,Cercis,canadensis,POINT (-75.15002 39.92168)
149990,151682,Quercus alba - white oak,Quercus alba,white oak,Quercus,alba,POINT (-75.22163 39.96638)
149991,151683,Quercus alba - white oak,Quercus alba,white oak,Quercus,alba,POINT (-75.2216 39.96626)
150009,151701,Acer rubrum - red maple,Acer rubrum,red maple,Acer,rubrum,POINT (-75.02411 40.08721)


In [438]:
genus_counts = genus_counts.rename(columns={"size": "Num_Genus_Obs"})
individuals = individuals.rename(columns={"count": "Num_Unique_Obs"})

In [439]:
merged = trees.merge(genus_counts, left_on="Genus", right_on="Genus", how="left" )
merged = merged.merge(individuals, left_on="scientific_name", right_on="scientific_name", how="left")
merged = merged.fillna({"nunique": 0, "Num_Unique_Obs": 0, "Num_Genus_Obs": 0 })


In [440]:
philly_count_species = trees.groupby("scientific_name", as_index=False).size()
philly_count_species = philly_count_species.rename(columns={"size": "Num_Species_Philly"})
philly_count_genus = trees.groupby("Genus", as_index=False).size()
philly_count_genus = philly_count_genus.rename(columns={"size": "Num_Genus_Philly"})
merged = merged.merge(philly_count_genus, left_on="Genus", right_on="Genus", how="left" )
merged = merged.merge(philly_count_species, left_on="scientific_name", right_on="scientific_name", how="left")
merged.head()

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,Num_Genus_Obs,nunique,Num_Unique_Obs,Num_Genus_Philly,Num_Species_Philly
0,1,Ginkgo biloba - ginkgo,Ginkgo biloba,ginkgo,Ginkgo,biloba,POINT (-75.2105 39.98383),6.0,2.0,6.0,3712,3712
1,2,Acer palmatum - japanese maple,Acer palmatum,japanese maple,Acer,palmatum,POINT (-75.21053 39.98374),4008.0,0.0,0.0,26852,339
2,3,Acer palmatum - japanese maple,Acer palmatum,japanese maple,Acer,palmatum,POINT (-75.21041 39.98376),4008.0,0.0,0.0,26852,339
3,4,Acer palmatum - japanese maple,Acer palmatum,japanese maple,Acer,palmatum,POINT (-75.2106 39.98395),4008.0,0.0,0.0,26852,339
4,5,Acer pseudoplatanus - sycamore maple,Acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus,POINT (-75.21028 39.98376),4008.0,4.0,22.0,26852,188


In [441]:
philly_species = merged.drop_duplicates(subset=["scientific_name"])
philly_species.head()

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,Num_Genus_Obs,nunique,Num_Unique_Obs,Num_Genus_Philly,Num_Species_Philly
0,1,Ginkgo biloba - ginkgo,Ginkgo biloba,ginkgo,Ginkgo,biloba,POINT (-75.2105 39.98383),6.0,2.0,6.0,3712,3712
1,2,Acer palmatum - japanese maple,Acer palmatum,japanese maple,Acer,palmatum,POINT (-75.21053 39.98374),4008.0,0.0,0.0,26852,339
4,5,Acer pseudoplatanus - sycamore maple,Acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus,POINT (-75.21028 39.98376),4008.0,4.0,22.0,26852,188
5,6,Ilex opaca - american holly,Ilex opaca,american holly,Ilex,opaca,POINT (-75.21047 39.98401),35.0,4.0,27.0,266,184
6,8,Koelreuteria paniculata - goldenrain tree,Koelreuteria paniculata,goldenrain tree,Koelreuteria,paniculata,POINT (-75.21008 39.98394),0.0,0.0,0.0,907,907


In [442]:
(
    alt.Chart(philly_species)
    .mark_circle()
    .encode(
    alt.X("Num_Unique_Obs:Q", scale=alt.Scale(type="symlog")),
    alt.Y("Num_Genus_Obs:Q", scale=alt.Scale(type="symlog")),
    size="Num_Species_Philly:Q",
    tooltip = ["scientific_name", "nunique", 'Num_Unique_Obs', "Num_Genus_Obs", "Num_Genus_Philly", "Num_Species_Philly"]
    )
    .properties(width=800, height=800)
    .interactive()
)

alt.Chart(...)

In [443]:
scaler = StandardScaler()
trees_scaled = scaler.fit_transform(philly_species[["nunique", 'Num_Unique_Obs', "Num_Genus_Obs", "Num_Genus_Philly", "Num_Species_Philly"]])

In [444]:
# Number of clusters to try out
n_clusters = list(range(2, 12))

# Run kmeans for each value of k
inertias = []
for k in n_clusters:
    
    # Initialize and run
    kmeans = KMeans(n_clusters=k, n_init=10)
    kmeans.fit(trees_scaled)
    
    # Save the "inertia"
    inertias.append(kmeans.inertia_)
# Initialize the knee algorithm
kn = KneeLocator(n_clusters, inertias, curve='convex', direction='decreasing')

# Print out the knee 
print(kn.knee)

5


In [445]:
kmeans = KMeans(n_clusters=4, n_init=12)
kmeans.fit(trees_scaled)
philly_species["label"] = kmeans.labels_

In [446]:
(
    alt.Chart(philly_species)
    .mark_circle()
    .encode(
        alt.X("Num_Unique_Obs:Q", scale=alt.Scale(type="symlog")),
        alt.Y("Num_Genus_Obs:Q", scale=alt.Scale(type="symlog")),
        size="Num_Species_Philly:Q",
        color=alt.Color("label:N", scale=alt.Scale(scheme="dark2")),
        tooltip=["scientific_name", "nunique", 'Num_Unique_Obs', "Num_Genus_Obs", "Num_Genus_Philly", "Num_Species_Philly"]
    )
    .properties(width=800, height=600)
    .interactive()
)

alt.Chart(...)

In [447]:
unique_genus = raw_npn.groupby("Genus", as_index=False)["Individual_ID"].nunique()
unique_genus


,Genus,Individual_ID
0,Abies,4
1,Acer,426
2,Achillea,7
3,Actaea,3
4,Aesculus,1
...,...,...
189,Vitis,2
190,Zea,2
191,Zelkova,11
192,Zizia,5


In [448]:
philly_genus = merged.drop_duplicates(subset="Genus")
philly_genus = philly_genus.merge(unique_genus, left_on="Genus", right_on="Genus", how="left")
philly_genus = philly_genus.drop(columns=["nunique"])
philly_genus = philly_genus.rename(columns={"Individual_ID": "nunique"})
philly_genus = philly_genus.fillna(0)

In [449]:
genus_scaled = scaler.fit_transform(philly_genus[["nunique", "Num_Genus_Obs", "Num_Genus_Philly"]])
# Run kmeans for each value of k
inertias = []
for k in n_clusters:
    
    # Initialize and run
    kmeans = KMeans(n_clusters=k, n_init=10)
    kmeans.fit(genus_scaled)
    
    # Save the "inertia"
    inertias.append(kmeans.inertia_)
# Initialize the knee algorithm
kn = KneeLocator(n_clusters, inertias, curve='convex', direction='decreasing')

# Print out the knee 
print(kn.knee)

4


In [450]:
kmeans = KMeans(n_clusters=4, n_init=12)
kmeans.fit(genus_scaled)
philly_genus["label"] = kmeans.labels_

In [451]:
(
    alt.Chart(philly_genus)
    .mark_circle()
    .encode(
        alt.X("Num_Genus_Obs:Q", scale=alt.Scale(type="symlog")),
        alt.Y("Num_Genus_Philly:Q", scale=alt.Scale(type="symlog")),
        size=alt.Size("nunique:Q", scale=alt.Scale(range=[25, 400])),
        color=alt.Color("label:N", scale=alt.Scale(scheme="dark2")),
        tooltip=["Genus", "nunique", "Num_Genus_Obs", "Num_Genus_Philly"]
    )
    .properties(width=800, height=600)
    .interactive()
)

alt.Chart(...)

In [452]:
#Num Genus per cluster
philly_genus.groupby("label").size()

label
0    87
1     2
2     4
3     2
dtype: int64

In [453]:
bad_data = philly_genus.loc[philly_genus["label"] == 0]

num_trees_bad_data = bad_data["Num_Genus_Philly"].sum()
print(num_trees_bad_data)


no_data = philly_genus.loc[philly_genus["nunique"] == 0]
num_trees_no_data = no_data["Num_Genus_Philly"].sum()
print(num_trees_no_data)

70870
7628


In [454]:
trees.loc[trees["Genus"] == "Sophora"]

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry
658,667,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.20933 39.98318)
660,669,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.2093 39.98313)
661,670,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.20939 39.98314)
662,671,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.20938 39.98306)
665,674,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.20942 39.9831)
...,...,...,...,...,...,...,...
141394,142832,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.25026 39.9058)
141400,142838,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.24226 39.91062)
143208,144649,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.21841 39.99266)
145679,147129,Sophora japonica - japanese pagoda tree,Sophora japonica,japanese pagoda tree,Sophora,japonica,POINT (-75.21792 39.91778)


In [455]:
raw_npn.loc[raw_npn["Species"].str.contains("japonic", case=False)].drop_duplicates(subset="Common_Name")

,Site_ID,Latitude,Longitude,Elevation_in_Meters,State,Species_ID,Genus,Species,Common_Name,Kingdom,Individual_ID,Phenophase_ID,Phenophase_Description,First_Yes_Year,First_Yes_Month,First_Yes_Day,First_Yes_DOY,First_Yes_Julian_Date,NumDays_Since_Prior_No,Last_Yes_Year,Last_Yes_Month,Last_Yes_Day,Last_Yes_DOY,Last_Yes_Julian_Date,NumDays_Until_Next_No,index_right,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,scientific_name,geometry
11325,46014,40.624527,-74.455261,76,NJ,184,Fallopia,japonica,Japanese knotweed,Plantae,281377,482,Initial growth (forbs),2021,12,11,345,2459560,35,2021,12,11,345,2459560,-9999,482,34,035,00882234,0500000US34035,34035,Somerset,Somerset County,NJ,New Jersey,06,781829190,7996119,Fallopia japonica,POINT (-74.45526 40.62453)
11591,57042,38.892887,-76.560081,12,MD,1247,Lonicera,japonica,Japanese honeysuckle,Plantae,353088,371,Breaking leaf buds,2021,1,14,14,2459229,-9999,2021,3,23,82,2459297,7,3192,24,003,01710958,0500000US24003,24003,Anne Arundel,Anne Arundel County,MD,Maryland,06,1074353889,448032843,Lonicera japonica,POINT (-76.56008 38.89289)


In [456]:
philly_species.head(2)

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,Num_Genus_Obs,nunique,Num_Unique_Obs,Num_Genus_Philly,Num_Species_Philly,label
0,1,Ginkgo biloba - ginkgo,Ginkgo biloba,ginkgo,Ginkgo,biloba,POINT (-75.2105 39.98383),6.0,2.0,6.0,3712,3712,0
1,2,Acer palmatum - japanese maple,Acer palmatum,japanese maple,Acer,palmatum,POINT (-75.21053 39.98374),4008.0,0.0,0.0,26852,339,2


In [457]:
trees_scaled = scaler.fit_transform(philly_species[["nunique", 'Num_Unique_Obs']])
kmeans = KMeans(n_clusters=4, n_init=12)
kmeans.fit(trees_scaled)
philly_species["label"] = kmeans.labels_
(
    alt.Chart(philly_species)
    .mark_circle()
    .encode(
        alt.Y("nunique:Q", scale=alt.Scale(type="symlog")),
        alt.X("Num_Unique_Obs:Q", scale=alt.Scale(type="symlog")),
        size="Num_Species_Philly:Q",
        color=alt.Color("label:N", scale=alt.Scale(scheme="dark2")),
        tooltip=["scientific_name", "nunique", 'Num_Unique_Obs', "Num_Genus_Obs", "Num_Genus_Philly", "Num_Species_Philly"]
    )
    .properties(width=800, height=600)
    .interactive()
)

alt.Chart(...)

In [458]:
philly_species.loc[(philly_species["nunique"] == 0) & (philly_species["Num_Unique_Obs"] == 0)]

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,Num_Genus_Obs,nunique,Num_Unique_Obs,Num_Genus_Philly,Num_Species_Philly,label
1,2,Acer palmatum - japanese maple,Acer palmatum,japanese maple,Acer,palmatum,POINT (-75.21053 39.98374),4008.0,0.0,0.0,26852,339,0
6,8,Koelreuteria paniculata - goldenrain tree,Koelreuteria paniculata,goldenrain tree,Koelreuteria,paniculata,POINT (-75.21008 39.98394),0.0,0.0,0.0,907,907,0
8,10,Amelanchier species - other serviceberry,Amelanchier species,other serviceberry,Amelanchier,species,POINT (-75.20996 39.98388),350.0,0.0,0.0,2194,1704,0
34,36,Prunus sargentii - sargent cherry,Prunus sargentii,sargent cherry,Prunus,sargentii,POINT (-75.12035 40.02201),362.0,0.0,0.0,16088,951,0
35,37,Gleditsia triacanthos inermis - thornless hone...,Gleditsia triacanthos inermis,thornless honeylocust,Gleditsia,triacanthos,POINT (-75.12101 40.02165),39.0,0.0,0.0,6292,1299,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
123992,125157,Sorbus aucuparia - european mountain ash,Sorbus aucuparia,european mountain ash,Sorbus,aucuparia,POINT (-75.17581 39.94451),0.0,0.0,0.0,51,4,0
126715,127967,Sorbus americana - american mountain ash,Sorbus americana,american mountain ash,Sorbus,americana,POINT (-75.20955 40.01983),0.0,0.0,0.0,51,6,0
140692,142126,Acer henryii – henrys maple,Acer henryii,henrys maple,Acer,henryii,POINT (-75.12051 40.04971),4008.0,0.0,0.0,26852,2,0
147979,149495,Ulmus alata - winged elm,Ulmus alata,winged elm,Ulmus,alata,POINT (-75.22588 40.03806),20.0,0.0,0.0,3525,1,0


In [459]:
raw_npn = raw_npn.drop(columns=["index_right", "STATEFP", "COUNTYFP", "COUNTYNS", "AFFGEOID", "GEOID", "NAME", "NAMELSAD", "STUSPS", "STATE_NAME", "LSAD", "ALAND", "AWATER"])

In [460]:
raw_npn

,Site_ID,Latitude,Longitude,Elevation_in_Meters,State,Species_ID,Genus,Species,Common_Name,Kingdom,Individual_ID,Phenophase_ID,Phenophase_Description,First_Yes_Year,First_Yes_Month,First_Yes_Day,First_Yes_DOY,First_Yes_Julian_Date,NumDays_Since_Prior_No,Last_Yes_Year,Last_Yes_Month,Last_Yes_Day,Last_Yes_DOY,Last_Yes_Julian_Date,NumDays_Until_Next_No,scientific_name,geometry
0,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,205,Open flowers (lilac),2010,4,7,97,2455294,1,2010,4,12,102,2455299,1,Syringa chinensis,POINT (-76.55337 38.88813)
1,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,206,Full flowering (lilac),2010,4,13,103,2455300,1,2010,4,19,109,2455306,7,Syringa chinensis,POINT (-76.55337 38.88813)
2,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,207,End of flowering (lilac/honeysuckle),2010,4,26,116,2455313,7,2010,4,28,118,2455315,2,Syringa chinensis,POINT (-76.55337 38.88813)
3,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,373,Breaking leaf buds (lilac/honeysuckle),2010,3,22,81,2455278,7,2010,3,22,81,2455278,4,Syringa chinensis,POINT (-76.55337 38.88813)
4,418,38.888126,-76.553368,24,MD,35,Syringa,chinensis,Red Rothomagensis lilac,Plantae,946,374,All leaf buds broken (lilac/honeysuckle),2010,3,26,85,2455282,4,2010,4,30,120,2455317,-9999,Syringa chinensis,POINT (-76.55337 38.88813)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27158,56856,40.052708,-75.687523,134,PA,1431,Amelanchier,arborea,common serviceberry,Plantae,375036,500,Flowers or flower buds,2026,4,8,98,2461139,-9999,2026,4,14,104,2461145,5,Amelanchier arborea,POINT (-75.68752 40.05271)
27159,56856,40.052708,-75.687523,134,PA,1431,Amelanchier,arborea,common serviceberry,Plantae,375036,501,Open flowers,2026,4,14,104,2461145,6,2026,4,14,104,2461145,5,Amelanchier arborea,POINT (-75.68752 40.05271)
27160,59544,38.996899,-77.112534,-9999,MD,195,Asclepias,tuberosa,butterfly milkweed,Plantae,375264,482,Initial growth (forbs),2026,4,12,102,2461143,2,2026,4,14,104,2461145,2,Asclepias tuberosa,POINT (-77.11253 38.9969)
27161,59544,38.996899,-77.112534,-9999,MD,195,Asclepias,tuberosa,butterfly milkweed,Plantae,375264,488,Leaves (forbs),2026,4,16,106,2461147,2,2026,4,20,110,2461151,-9999,Asclepias tuberosa,POINT (-77.11253 38.9969)


In [461]:
npn_genus_in_philly = raw_npn.loc[raw_npn["Genus"].isin(trees["Genus"])]
npn_genus_in_philly.groupby("Genus", as_index=False)["Individual_ID"].nunique().sort_values(by="Individual_ID")

,Genus,Individual_ID
36,Metasequoia,1
2,Aesculus,1
40,Picea,1
13,Chamaecyparis,1
20,Euonymus,1
54,Tilia,1
15,Cladrastis,1
25,Gymnocladus,1
18,Crataegus,2
23,Ginkgo,2


In [462]:
unique_genus

,Genus,Individual_ID
0,Abies,4
1,Acer,426
2,Achillea,7
3,Actaea,3
4,Aesculus,1
...,...,...
189,Vitis,2
190,Zea,2
191,Zelkova,11
192,Zizia,5


In [480]:
genus_activity = pd.read_csv("../data/processed/genus_activity_curves.csv")
phenophase_definitions = pd.read_csv("../data/raw/phenophase_definitions.csv")
phenophase_definitions = phenophase_definitions[["Phenophase_ID", "Phenophase_Name"]]
phenophase_definitions = phenophase_definitions.drop_duplicates(subset=["Phenophase_ID"], keep="last")
genus_activity.groupby("Genus").size()
phenophase_definitions

,Phenophase_ID,Phenophase_Name
0,56,First leaf
1,57,75% leaf elongation
2,58,First flower
3,59,Last flower
4,60,First fruit ripe
...,...,...
395,467,Early season leaf expansion
396,393,Ripe seed cones
397,490,Pollen cones
398,495,Open pollen cones


In [482]:
genus_activity_merged = genus_activity.merge(unique_genus, how="left", right_on="Genus", left_on="Genus").fillna(0)

genus_activity_merged = genus_activity_merged.rename(columns={"Individual_ID": "Num_Individuals"})

genus_activity_merged = genus_activity_merged.merge(phenophase_definitions, how="left", right_on="Phenophase_ID", left_on="Phenophase_ID")
genus_activity_merged.head(20)
genus_activty_over5_individuals = genus_activity_merged.loc[genus_activity_merged["Num_Individuals"] >= 5]



In [483]:
genus_activty_over5_individuals
phl_genus_over5 = trees.loc[trees["Genus"].isin(genus_activty_over5_individuals["Genus"])]
print(len(phl_genus_over5) / len (trees))

0.8330856133568834


In [486]:
genus_activty_over5_individuals.head(1)


,Genus,Phenophase_ID,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,Num_Individuals,Phenophase_Name
8,Acer,180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.469484,0.0,0.0,0.0,0.0,0.0,0.234742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,426,>=75% of full leaf size


In [487]:
cols_to_front = ["Genus", "Num_Individuals", "Phenophase_ID", "Phenophase_Name"]
new_order = cols_to_front + [col for col in genus_activty_over5_individuals.columns if col not in cols_to_front]
genus_activty_over5_individuals = genus_activty_over5_individuals[new_order]
genus_activty_over5_individuals

,Genus,Num_Individuals,Phenophase_ID,Phenophase_Name,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366
8,Acer,426,180,>=75% of full leaf size,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.234742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.469484,0.0,0.0,0.0,0.0,0.0,0.234742,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,Acer,426,181,>=50% of leaves colored,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0